In [2]:
# Building RAG Application with LangChain and Ollama
# 0 : Installing Required Libraries

In [ ]:
!pip install langchain-community pypdf
!pip install -qU langchain-ollama
!pip install -qU chromadb langchain-chroma langchain

In [ ]:
# 1 : Loading a PDF File

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Think-And-Grow-Rich_2011-06.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

In [ ]:
# 2 : Chunking the document

In [ ]:
# splitting the document into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

In [ ]:
# generating embeddings for the document chunks

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="llama3.1:latest")

In [ ]:
# saving embeddings to Chroma vector DB

In [ ]:
from langchain_chroma import Chroma

persist_dir = "./chroma_db"
collection_name = "example_collection"

# Newer Chroma no longer requires constructing chromadb.Client with legacy Settings.
# Let the LangChain Chroma wrapper manage the client (it will create the collection if needed).
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=persist_dir,
)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [ ]:


from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a book. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, tools, system_prompt=prompt)
query = (
     "burning desire as a driving force"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

In [ ]:
# DESIRED OUPUT BELOW  :NOTE -> YOU CAN USE YOUR OWN PDF'S FOR INPUT DATA ?

In [ ]:
================================ Human Message =================================

burning desire as a driving force
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (4150591f-def0-453f-9250-3ba55099316e)
 Call ID: 4150591f-def0-453f-9250-3ba55099316e
  Args:
    query: burning desire as a driving force in personal growth and motivation
================================= Tool Message =================================
Name: retrieve_context

Source: {'author': 'Brod', 'keywords': 'free, ebook, think and grow rich, make money online, home based business, opportunity', 'creator': 'Writer', 'producer': 'OpenOffice.org 3.2', 'total_pages': 253, 'page': 95, 'creationdate': '2011-06-06T18:53:08+01:00', 'start_index': 1582, 'subject': 'Free Digital Download PDF eBook Edition', 'source': 'Think-And-Grow-Rich_2011-06.pdf', 'page_label': '96', 'title': 'Think and Grow Rich by Napoleon Hill'}
Content: didn't you reach that decision a long time ago? The money was waiting for you 
all the time!'
"Things began to happen in a hurry. I called the newspapers and 
announced I would preach a sermon the following morning, entitled, 'What I 
would do if I had a Million Dollars.'
"I went to work on the sermon immediately, but I must tell you, frankly, 
the task was not difficult, because I had been preparing that sermon for almost 
two years. The spirit back of it was a part of me! 
"Long before midnight I had finished writing the sermon. I went to bed 
and slept with a feeling of confidence, for I could see myself already in. 
possession of the million dollars. 
"Next morning I arose early, went into the bathroom, read the sermon, 
then knelt on my knees and asked that my sermon might come to the attention of 
someone who would supply the needed money.
Do You Feel Frustrated, Stuck Or Overwhelmed by the Concepts in this eBook?

Source: {'page': 28, 'keywords': 'free, ebook, think and grow rich, make money online, home based business, opportunity', 'producer': 'OpenOffice.org 3.2', 'creationdate': '2011-06-06T18:53:08+01:00', 'start_index': 1581, 'page_label': '29', 'source': 'Think-And-Grow-Rich_2011-06.pdf', 'subject': 'Free Digital Download PDF eBook Edition', 'total_pages': 253, 'author': 'Brod', 'title': 'Think and Grow Rich by Napoleon Hill', 'creator': 'Writer'}
Content: spirit of open-mindedness, and remember as you read, they are the invention of 
no one man. The principles were gathered from the life experiences of more than 
500 men who actually accumulated riches in huge amounts; men who began in 
poverty, with but little education, without influence. The principles worked for 
these men. You can put them to work for your own enduring benefit. 
You will find it easy, not hard, to do.
Before you read the next chapter, I want you to know that it conveys 
factual information which might easily change your entire financial destiny, as it 
has so definitely brought changes of stupendous proportions to two people 
described. 
I want you to know, also, that the relationship between these two men and 
myself, is such that I could have taken no liberties with the facts, even if I had 
wished to do so. One of them has been my closest personal friend for almost 
Do You Feel Frustrated, Stuck Or Overwhelmed by the Concepts in this eBook?
================================== Ai Message ==================================

Based on the tool's output, it appears that a burning desire can be a powerful driving force for personal growth and motivation. Napoleon Hill, the author of "Think and Grow Rich," emphasizes the importance of having a clear and focused goal, which he refers to as a "burning desire." He shares examples of individuals who have achieved success by cultivating this desire and taking consistent action towards their goals.

To answer your original question, a burning desire can be a driving force for personal growth and motivation when it is:

1. Clearly defined: Having a specific and well-defined goal in mind helps to focus one's efforts and energies.
2. Focused on the positive: A burning desire should be focused on what you want to achieve, rather than what you want to avoid.
3. Strongly felt: The desire should be intense and deeply felt, driving you to take action towards your goals.
4. Consistently pursued: Cultivating a burning desire requires consistent effort and dedication over time.

By cultivating a burning desire and taking consistent action towards your goals, you can tap into the power of motivation and drive personal growth and success.
 